# Creating artificial storm

In [12]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split         

# Importing data needed to compute the average wavelet power and size

In [13]:
test = pd.read_csv("../../output/data/dakar/data-test-eps-dakar.csv")
train = pd.read_csv("../../output/data/dakar/data-train-eps-dakar.csv")
train, validation = train_test_split(train, test_size=0.2, random_state=12)

In [14]:
def choose_header(n, location, lead_time):
    """
    Generate header names for n closest storms.

    Returns:
        tuple of two lists:
            - input headers: year, month, day, hour, minute, lat1..n, lon1..n, wp1..n, size1..n, d1..n, mask1..n
            - target header: Cb_{location}_t{lead_time}
    """
    input_headers = ['year', 'month', 'day', 'hour', 'minute']
    
    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        input_headers.extend([f'{prefix}{i}' for i in range(1, n + 1)])
        
    return input_headers

In [15]:
def haversine_distance(lat1, lon1, lat2, lon2):
        """
        Compute Haversine distance between two points or arrays of points.
        Inputs are in degrees. Output is in kilometers.
        
        Supports both scalar and array inputs (NumPy).
        """
        R = 6371.0  # Earth radius in kilometers

        # Convert degrees to radians
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)

        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad

        a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

        return R * c

In [16]:
def generate_fictional_storm(city_lat, city_lon):
    """
    Generate a fictional storm located more than 1000 km away from the given city.
    """
    min_dist_km = 1000
    # Extended context domain (centered in Dakar, min distance ~1000 km)
    EXTENDED_CONTEXT_DOMAIN_LAT_MIN, EXTENDED_CONTEXT_DOMAIN_LAT_MAX = 14.69 - 9, 14.69 + 9    # 5.69 to 23.69
    EXTENDED_CONTEXT_DOMAIN_LON_MIN, EXTENDED_CONTEXT_DOMAIN_LON_MAX = -17.45 - 9.3, -17.45 + 9.3  # -26.75 to -8.15

    while True:
        lat = np.random.uniform(EXTENDED_CONTEXT_DOMAIN_LAT_MIN, EXTENDED_CONTEXT_DOMAIN_LAT_MAX)
        lon = np.random.uniform(EXTENDED_CONTEXT_DOMAIN_LON_MIN, EXTENDED_CONTEXT_DOMAIN_LON_MAX)
        distance = haversine_distance(lat, lon, city_lat, city_lon)
        if distance > min_dist_km:
            return {
                'lat': lat,
                'lon': lon,
                'wp': 0.0,
                'size': 0,
                'distance': distance,
                'mask': 0
            }

In [17]:
Dakar_lon = -17.467686
Dakar_lat = 14.716677

In [18]:
# Get all wp and size values as flattened arrays
wp_values = train[[f"wp{i}" for i in range(1, 4)]].values.flatten()
size_values = train[[f"size{i}" for i in range(1, 4)]].values.flatten()

# Remove zeros before averaging
avg_wp = wp_values[wp_values > 0].mean()
avg_size = size_values[size_values > 0].mean()

In [19]:
year = 2020
month = 9
day = 5
hour = 15
minute = 0

In [20]:
# Define resolution (change as needed)
lat_resolution = lon_resolution = 0.1


CONTEXT_DOMAIN_LAT_MIN, CONTEXT_DOMAIN_LAT_MAX = 8.69, 20.69
CONTEXT_DOMAIN_LON_MIN, CONTEXT_DOMAIN_LON_MAX = -23.45, -11.45


# Create arrays of lats and lons
lats = np.arange(CONTEXT_DOMAIN_LAT_MIN, CONTEXT_DOMAIN_LAT_MAX + lat_resolution, lat_resolution)
lons = np.arange(CONTEXT_DOMAIN_LON_MIN, CONTEXT_DOMAIN_LON_MAX + lon_resolution, lon_resolution)

# Create meshgrid
lon_grid, lat_grid = np.meshgrid(lons, lats)

In [21]:
lat_points = lat_grid.flatten()
lon_points = lon_grid.flatten()

In [22]:
generated_data = []

for lat, lon in zip(lat_grid.flatten(), lon_grid.flatten()):
    storm0 = {
        'lat': lat,
        'lon': lon,
        'wp': avg_wp,
        'size': avg_size,
        'distance': haversine_distance(lat, lon, Dakar_lat, Dakar_lon),
        'mask': 1
    }

    storm1 = generate_fictional_storm(Dakar_lat, Dakar_lon)
    storm2 = generate_fictional_storm(Dakar_lat, Dakar_lon)

    storm_data = [storm0, storm1, storm2]

    entry = {
        'year': year,
        'month': month,
        'day': day,
        'hour': hour,
        'minute': minute
    }

    # Add lat1..3, lon1..3, wp1..3, size1..3, d1..3, mask1..3 in order
    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        for i, storm in enumerate(storm_data, start=1):
            key = f"{prefix}{i}"
            if prefix == 'd':
                entry[key] = storm['distance']
            else:
                entry[key] = storm[prefix]

    generated_data.append(entry)

# pd.DataFrame(generated_data)
pd.DataFrame(generated_data).to_csv(f'./data/single/artificial-data-{lat_resolution}-1500.csv', index=False)